# 00 - Quality Checks

Reusable data quality functions used by Silver and Gold.

The checks are intentionally implemented in a notebook because the assessment feedback asked for a dedicated quality notebook. The layer notebooks execute this file with `%run` and then build their own quality reports.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F
from config import BASE_PARQUET_PATH, FAIL_ON_CRITICAL
from utils import table_path

In [ ]:
QUARANTINE_PATH = str(Path(BASE_PARQUET_PATH) / "quarantine")

In [ ]:
def quality_result(layer, table_name, check_name, severity, failed_count, rule_description):
    status = "PASS" if failed_count == 0 else "FAIL"
    return {
        "layer": layer,
        "table_name": table_name,
        "check_name": check_name,
        "severity": severity,
        "status": status,
        "failed_count": int(failed_count),
        "rule_description": rule_description,
    }

In [ ]:
def check_primary_key_unique(df, layer, table_name, pk_column, severity="CRITICAL"):
    total_rows = df.count()
    distinct_pk = df.select(pk_column).where(F.col(pk_column).isNotNull()).distinct().count()
    null_pk = df.filter(F.col(pk_column).isNull()).count()
    duplicate_count = total_rows - distinct_pk - null_pk
    failed_count = duplicate_count + null_pk

    return quality_result(
        layer=layer,
        table_name=table_name,
        check_name=f"primary_key_unique__{pk_column}",
        severity=severity,
        failed_count=failed_count,
        rule_description=f"{pk_column} must be unique and not null. Duplicates and nulls are not allowed."
    )

In [ ]:
def check_not_null(df, layer, table_name, column_name, severity="CRITICAL"):
    failed_count = df.filter(F.col(column_name).isNull()).count()

    return quality_result(
        layer=layer,
        table_name=table_name,
        check_name=f"not_null__{column_name}",
        severity=severity,
        failed_count=failed_count,
        rule_description=f"{column_name} must not be null."
    )

In [ ]:
def check_non_negative(df, layer, table_name, column_name, severity="CRITICAL"):
    failed_count = df.filter(F.col(column_name) < 0).count()

    return quality_result(
        layer=layer,
        table_name=table_name,
        check_name=f"non_negative__{column_name}",
        severity=severity,
        failed_count=failed_count,
        rule_description=f"{column_name} must be greater than or equal to zero."
    )

In [ ]:
def check_foreign_key_exists(left_df, right_df, layer, table_name, fk_column, right_column, severity="CRITICAL"):
    failed_count = (
        left_df
        .select(fk_column)
        .where(F.col(fk_column).isNotNull())
        .distinct()
        .join(
            right_df.select(F.col(right_column).alias(fk_column)).where(F.col(fk_column).isNotNull()).distinct(),
            on=fk_column,
            how="left_anti"
        )
        .count()
    )

    return quality_result(
        layer=layer,
        table_name=table_name,
        check_name=f"foreign_key_exists__{fk_column}",
        severity=severity,
        failed_count=failed_count,
        rule_description=f"Every {fk_column} must exist in the referenced table."
    )

In [ ]:
def check_negative_date_interval(df, layer, table_name, start_date_col, end_date_col, severity="CRITICAL"):
    failed_count = (
        df
        .filter(
            F.col(start_date_col).isNotNull()
            & F.col(end_date_col).isNotNull()
            & (F.col(end_date_col) < F.col(start_date_col))
        )
        .count()
    )

    return quality_result(
        layer=layer,
        table_name=table_name,
        check_name=f"negative_date_interval__{end_date_col}_before_{start_date_col}",
        severity=severity,
        failed_count=failed_count,
        rule_description=f"{end_date_col} must not be earlier than {start_date_col}."
    )

In [ ]:
def quarantine_negative_dates(df, table_name, start_date_col="OrderDate", end_date_col="ShipDate"):
    invalid_df = (
        df
        .filter(
            F.col(start_date_col).isNotNull()
            & F.col(end_date_col).isNotNull()
            & (F.col(end_date_col) < F.col(start_date_col))
        )
    )

    invalid_count = invalid_df.count()

    if invalid_count > 0:
        (
            invalid_df.write
            .format("parquet")
            .mode("overwrite")
            .save(str(Path(QUARANTINE_PATH) / "negative_dates" / table_name))
        )

    return invalid_count

In [ ]:
def build_quality_report(spark, results):
    return spark.createDataFrame(results)


def evaluate_quality_report(quality_df):
    critical_failures = (
        quality_df
        .filter((F.col("severity") == "CRITICAL") & (F.col("status") == "FAIL"))
        .count()
    )

    if FAIL_ON_CRITICAL and critical_failures > 0:
        raise Exception(f"Critical data quality checks failed: {critical_failures}")

    return critical_failures